This builds the bar chart comparing coverage between VAMP-seq and SGE assays

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from matplotlib.patches import Polygon

In [ ]:
#Paths to each of the SGE and VAMP-seq subsets from the main dataframe
sge_ppj_subset = pd.read_excel('../Data/filtered_ppj_data/20251202_SGEsubset.xlsx')
vampseq_ppj_subset = pd.read_excel('../Data/filtered_ppj_data/20251204_VAMPseqsubset_wDups.xlsx')

In [ ]:
def read_data(sge, vampseq): #Reads all data and changes variant consequence annotations to be more human readable
 
    sge_genes = ['BARD1', 'PALB2', 'BRCA2', 'RAD51D', 'XRCC2', 'CTCF', 'SFPQ'] #SGE genes

    vampseq_genes = ['G6PD', 'TSC2', 'F9'] #VAMP-seq genes
    
    sge_dict = {} #Empty dictionary to store SGE data in form or {gene_name: gene_data}

    for gene in sge_genes:
        print(f' Reading {gene}...')
        df = sge.loc[sge['Gene'] == gene]
        #print(df)
        df = df.loc[df['ref_allele'].str.len() == 1]
        df = df.dropna(subset = ['simplified_consequence'])
        df = df.rename(columns = {'hg38_start': 'pos'})
        df['amino_acid_change'] = df['aa_ref'] + df['aa_pos'].astype(str) + df['aa_alt']
        
        sge_dict[gene] = df


    #Analagous code for reading VAMP-seq data
    vamp_dict = {}
    for gene in vampseq_genes:
        print(f' Reading {gene}...')
        
        #Merges TSC2 datasets into single dataframe
        if gene == 'TSC2': 
            df = vampseq.loc[vampseq['Gene'] == gene]

            rapgap_df = df.loc[df['Dataset'].str.contains('rapgap')].copy()
            tuberin_df = df.loc[df['Dataset'].str.contains('tuberin')].copy()
            tsc2_sets = [('TSC2 RapGAP', rapgap_df), ('TSC2 Tuberin', tuberin_df)]

            for set in tsc2_sets:
                name, df = set
                df = df.drop_duplicates(subset = ['ID'])
                df = df.rename(columns = {'hg38_start': 'pos'})
                df['amino_acid_change'] = df['aa_ref'] + df['aa_pos'].astype(str) + df['aa_alt']
                vamp_dict[name] = df
        else:
            df = vampseq.loc[vampseq['Gene'] == gene]

            df = df.drop_duplicates(subset = ['ID'])
            df = df.rename(columns = {'hg38_start': 'pos'})
            df['amino_acid_change'] = df['aa_ref'] + df['aa_pos'].astype(str) + df['aa_alt']
        
            vamp_dict[gene] = df

    #print(sge_dict)
    return sge_dict, vamp_dict

In [ ]:
def assay_comparison_bars(sge_data, vampseq_data):
    # Set font - use Arial as Helvetica often lacks bold variant in matplotlib
    plt.rcParams['font.family'] = 'Arial'  # Or try 'Helvetica Neue'
    
    # Calculate totals   
    sge_total_genomic = sum([len(set(sge_data[g]['pos'])) for g in sge_data])
    sge_total_amino = sum([sge_data[g]['amino_acid_change'].nunique() for g in sge_data])
    
    vamp_total_genomic = 0
    vamp_total_amino = 0
    for gene in vampseq_data:
        df = vampseq_data[gene]
        vamp_total_genomic += len(set(df['aa_pos'])) * 3
        vamp_total_amino += len(set(df['amino_acid_change']))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 5))
    
    assays = ['VAMP-seq', 'SGE']
    x = np.array([0, 0.6])
    width = 0.4
    colors = ['#F4A261', '#A8DADC']
    
    # Left plot: Genomic positions
    genomic_values = [vamp_total_genomic, sge_total_genomic]
    bars1 = ax1.bar(x, genomic_values, width, color=colors, alpha=0.9)
    ax1.set_title('Genomic Positions', fontsize=16, fontweight='bold', pad=15)
    ax1.set_xticks(x)
    ax1.set_xticklabels(assays, fontsize=12, fontweight='bold')
    ax1.set_xlim(-0.35, 0.95)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.tick_params(left=False, labelleft=False)  # Keep ticks, remove labels
    
    # Add value labels
    for i, v in enumerate(genomic_values):
        ax1.text(x[i], v + max(genomic_values)*0.02, f'{v:,}', ha='center', va='bottom', 
                 fontsize=11, fontweight='bold')
    
    # Right plot: Amino acid changes
    amino_values = [vamp_total_amino, sge_total_amino]
    bars2 = ax2.bar(x, amino_values, width, color=colors, alpha=0.9)
    ax2.set_title('Amino Acid Changes', fontsize=16, fontweight='bold', pad=15)
    ax2.set_xticks(x)
    ax2.set_xticklabels(assays, fontsize=12, fontweight='bold')
    ax2.set_xlim(-0.35, 0.95)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.tick_params(left=False, labelleft=False)  # Keep ticks, remove labels
    
    # Add value labels
    for i, v in enumerate(amino_values):
        ax2.text(x[i], v + max(amino_values)*0.02, f'{v:,}', ha='center', va='bottom', 
                 fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    #fig.savefig('/Users/ivan/Desktop/pillar_project_figs/20260120_vampseq_sge_bars.svg')

In [ ]:
def main():
    sge_data, vampseq_data = read_data(sge_ppj_subset, vampseq_ppj_subset)

    assay_comparison_bars(sge_data, vampseq_data)

In [ ]:
main()